In [4]:
# Import the data and libraries 
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
import os
import sys
sys.path.append("..")


df = pd.read_csv("G:/DS/06-03-2026/Excel/Messy/Messy_HR_Dataset.csv")
# Check missing values
print(df.isnull().sum())

Employee_ID           0
First_Name            0
Last_Name             0
Age                  18
Gender                0
Department            0
City                  0
Salary                8
Experience_Years     17
Performance_Score    17
Education             0
Join_Date             0
Remote_Work           0
Bonus                22
dtype: int64


In [5]:
# Handling missing values
# Age
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age'] = df['Age'].apply(lambda x: x if 18 <= x <= 65 else np.nan)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Salary
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

# Experience
df['Experience_Years'] = pd.to_numeric(df['Experience_Years'], errors='coerce')
df['Experience_Years'] = df['Experience_Years'].fillna(df['Experience_Years'].median())

# Performance Score
df['Performance_Score'] = pd.to_numeric(df['Performance_Score'], errors='coerce')
df['Performance_Score'] = df['Performance_Score'].fillna(df['Performance_Score'].median())

# Bonus
df['Bonus'] = pd.to_numeric(df['Bonus'], errors='coerce')
df['Bonus'] = df['Bonus'].fillna(df['Bonus'].median())

# Gender Correction
df['Gender'] = df['Gender'].str.replace('.', '', regex=False).str.strip().str.lower()
df['Gender'] = df['Gender'].str[0]
df['Gender'] = df['Gender'].map({'f': 'Female', 'm': 'Male'})

# Department Correction
df['Department']=df['Department'].str.upper().str.strip()
# City Correct
df['City']= df['City'].str.upper().str.strip()
# Remote Work Correction
df['Remote_Work'] = df['Remote_Work'].apply(lambda x: 'Yes' if str(x).startswith('Y') else 'No')

# Check missing values
print(df.isnull().sum())
print(df.info())


Employee_ID          0
First_Name           0
Last_Name            0
Age                  0
Gender               0
Department           0
City                 0
Salary               0
Experience_Years     0
Performance_Score    0
Education            0
Join_Date            0
Remote_Work          0
Bonus                0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        210 non-null    str    
 1   First_Name         210 non-null    str    
 2   Last_Name          210 non-null    str    
 3   Age                210 non-null    float64
 4   Gender             210 non-null    str    
 5   Department         210 non-null    str    
 6   City               210 non-null    str    
 7   Salary             210 non-null    float64
 8   Experience_Years   210 non-null    float64
 9   Performance_Score  210 non-null    flo

In [6]:
# Removing Duplicte
# Duplicate Removal
df=df.drop_duplicates(subset=['Employee_ID'])

In [7]:
# Identifying Outliers and treatment
numeric_cols = ['Salary','Bonus','Experience_Years','Performance_Score']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col} outliers before treatment:", len(outliers))

    df[col] = df[col].clip(lower, upper)

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col} outliers after treatment:", len(outliers))

print(df['Salary'].describe())


Salary outliers before treatment: 2
Salary outliers after treatment: 0
Bonus outliers before treatment: 0
Bonus outliers after treatment: 0
Experience_Years outliers before treatment: 0
Experience_Years outliers after treatment: 0
Performance_Score outliers before treatment: 0
Performance_Score outliers after treatment: 0
count       200.000000
mean      75536.808063
std       28510.732483
min         500.000000
25%       56447.530000
50%       78426.670000
75%       96367.772500
max      156248.136250
Name: Salary, dtype: float64


In [8]:
# Normalization and Standardization andfeature scaling 
cols = ['Age','Salary','Bonus','Experience_Years','Performance_Score']
scaler = MinMaxScaler()
df_normalized = df.copy() 
df_normalized[cols] = scaler.fit_transform(df_normalized[cols])
print(df_normalized[cols].head())
print(df_normalized[cols].describe())
print("="*45)


# Standardization
scaler = StandardScaler()
df_standardized = df.copy()
df_standardized[cols] = scaler.fit_transform(df_standardized[cols])

print(df_standardized[cols].head())

print("="*45)


# Encoding Categorical Variables
categorical_cols = ['Gender','Department','City','Education','Remote_Work']
df_encoded = pd.get_dummies(df, columns=categorical_cols,drop_first=True,dtype=int)
print(df_encoded.head())

        Age    Salary     Bonus  Experience_Years  Performance_Score
0  0.527778  0.570064  0.482829          0.485714           0.494382
1  0.138889  0.594048  0.811296          0.971429           0.415730
2  0.750000  0.213876  0.808980          0.171429           0.348315
3  0.194444  0.697395  0.774704          0.657143           0.573034
4  0.166667  0.374006  0.534261          0.485714           0.696629
              Age      Salary       Bonus  Experience_Years  Performance_Score
count  200.000000  200.000000  200.000000        200.000000         200.000000
mean     0.522639    0.481783    0.495635          0.484143           0.497978
std      0.288111    0.183057    0.268978          0.285027           0.272170
min      0.000000    0.000000    0.000000          0.000000           0.000000
25%      0.270833    0.359218    0.283444          0.250000           0.280899
50%      0.527778    0.500338    0.482829          0.485714           0.494382
75%      0.750000    0.615531    

In [9]:
# Save cleaned dataset
df.to_csv("G:/DS/06-03-2026/Excel/Clean/Cleaned_Mesy_HR_Dataset.csv", index=False)
df_normalized.to_csv("G:/DS/06-03-2026/Excel/Clean/Normalized_HR_Dataset.csv", index=False)
df_standardized.to_csv("G:/DS/06-03-2026/Excel/Clean/Standardized_HR_Dataset.csv", index=False)
df_encoded.to_csv("G:/DS/06-03-2026/Excel/Clean/Final_Preprocessed_HR_Dataset.csv", index=False)
